# 2.1 Determining a person's gender and age from a video

This cell uses MiVOLO v2. From this directory, install the dependencies before running the notebook:

```bash
python -m pip install setuptools==80.10.2
python -m pip install --no-build-isolation -r requirements.txt
```

The first run downloads the model weights from Hugging Face.

In [2]:
import cv2
import torch
from transformers import AutoConfig, AutoImageProcessor, AutoModel, AutoModelForImageClassification

mivolo_revision = "53393526c220e34cdd7b722b36d22b6f9e5f4241"
detector_revision = "b3d071aedccd46a3b2d4b40609da6880a815f395"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_id = "iitolstykh/mivolo_v2"
model_config = AutoConfig.from_pretrained(
    model_id, trust_remote_code=True, revision=mivolo_revision
)
image_processor = AutoImageProcessor.from_pretrained(
    model_id, trust_remote_code=True, revision=mivolo_revision
)
age_gender_model = AutoModelForImageClassification.from_pretrained(
    model_id,
    trust_remote_code=True,
    revision=mivolo_revision,
    dtype=torch.float32,
).to(device).eval()
face_person_detector = AutoModel.from_pretrained(
    "iitolstykh/YOLO-Face-Person-Detector",
    trust_remote_code=True,
    revision=detector_revision,
    dtype=torch.float32,
)

def analyze_frame(frame):
    detections = face_person_detector(
        frame, conf=0.4, iou=0.7, verbose=False, device=str(device)
    )[0]
    face_boxes = []
    person_boxes = []
    frame_height, frame_width = frame.shape[:2]

    for detection in detections.boxes:
        label = detections.names[int(detection.cls.item())]
        x1, y1, x2, y2 = [int(round(value)) for value in detection.xyxy[0].tolist()]
        box = (
            max(0, min(x1, frame_width)),
            max(0, min(y1, frame_height)),
            max(0, min(x2, frame_width)),
            max(0, min(y2, frame_height)),
        )
        if box[2] <= box[0] or box[3] <= box[1]:
            continue
        if label == "face":
            face_boxes.append(box)
        elif label == "person":
            person_boxes.append(box)

    face_crops = []
    person_crops = []
    matched_face_boxes = []
    for face_box in face_boxes:
        face_x1, face_y1, face_x2, face_y2 = face_box
        center_x = (face_x1 + face_x2) / 2
        center_y = (face_y1 + face_y2) / 2
        matching_people = [
            box
            for box in person_boxes
            if box[0] <= center_x <= box[2] and box[1] <= center_y <= box[3]
        ]
        if not matching_people:
            continue
        person_box = min(
            matching_people,
            key=lambda box: (box[2] - box[0]) * (box[3] - box[1]),
        )
        person_x1, person_y1, person_x2, person_y2 = person_box
        face_crops.append(frame[face_y1:face_y2, face_x1:face_x2])
        person_crops.append(frame[person_y1:person_y2, person_x1:person_x2])
        matched_face_boxes.append(face_box)

    if not face_crops:
        return []

    faces_input = image_processor(images=face_crops)[
        "pixel_values"
    ].to(device=device, dtype=age_gender_model.dtype)
    body_input = image_processor(images=person_crops)[
        "pixel_values"
    ].to(device=device, dtype=age_gender_model.dtype)
    with torch.inference_mode():
        output = age_gender_model(faces_input=faces_input, body_input=body_input)

    predictions = []
    for index, face_box in enumerate(matched_face_boxes):
        gender_index = int(output.gender_class_idx[index].item())
        gender = model_config.gender_id2label[gender_index]
        age = float(output.age_output[index].item())
        predictions.append((face_box, age, gender))
    return predictions


camera = cv2.VideoCapture(0)
if not camera.isOpened():
    camera.release()
    raise RuntimeError("Cannot open the webcam. Check the connection and permissions.")

camera.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
camera.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*"MJPG"))
actual_width = int(camera.get(cv2.CAP_PROP_FRAME_WIDTH))
actual_height = int(camera.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"Camera resolution: {actual_width}x{actual_height}")

analysis_interval = 15
frame_number = 0
predictions = []
window_name = "Age and gender estimation"
print("Age and gender estimation is based on appearance. Press q in the video window to exit.")

try:
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    frame_width = int(camera.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(camera.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cv2.resizeWindow(window_name, frame_width, frame_height)
    while True:
        success, frame = camera.read()
        if not success:
            raise RuntimeError("Failed to capture a frame from the webcam.")

        frame_number += 1
        if frame_number == 1 or frame_number % analysis_interval == 0:
            predictions = analyze_frame(frame)

        if not predictions:
            cv2.putText(
                frame,
                "Face/person not detected",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 0, 255),
                2,
            )

        for face_box, age, gender in predictions:
            x1, y1, x2, y2 = face_box
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            label = f"Age ~ {age:.0f}, {gender}"
            cv2.putText(
                frame,
                label,
                (x1, max(y1 - 10, 25)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2,
            )

        cv2.imshow(window_name, frame)
        key = cv2.waitKey(1) & 0xFF
        try:
            window_open = cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) >= 1
        except cv2.error:
            window_open = False
        if key == ord("q") or not window_open:
            break
finally:
    camera.release()
    cv2.destroyAllWindows()

Overriding model.yaml nc=80 with nc=2
YOLOv8x summary (fused): 268 layers, 68125494 parameters, 0 gradients, 257.4 GFLOPs
Camera resolution: 1920x1080
Age and gender estimation is based on appearance. Press q in the video window to exit.
